# PARC2026 π0.5 Full LoRA Training — A100 / resume-safe

固定Simulator評価とtop-2 tie-breakで選んだ group-aware dataset manifest を使い、π0.5 LoRAを20k optimizer steps学習するNotebookです。

- **A100 40GB/80GB または H100 向け**です。L4は固定Simulator評価用で、長時間本学習では使用しません。
- Driveにprefetch済みの `lerobot/libero_plus` datasetを `/content` にstageします。
- group-aware schema-v2 manifestとexact/action leakage=0を再検証してから学習します。
- A100 40GBは BS=8 / GA=16、60GB以上は BS=16 / GA=8（effective batch=128）を自動選択します。
- 500 optimizer stepsごとに保存し、安定した `checkpoints/last` をDriveへmirrorします。Colab runtimeが切れた場合も、新しいA100 runtimeで **Run all** すれば最新Drive checkpointからresumeします。
- top-2 tie-breakがまだ未完了なら安全に停止します。手動overrideは明示的に設定した場合だけ有効です。


In [ ]:
# 0/2 — Fresh A100 runtime preflight + Drive + HF token
import os, shutil, subprocess
from pathlib import Path

print('=== FULL TRAIN NOTEBOOK PREFLIGHT ===', flush=True)

try:
    from google.colab import drive, userdata
except Exception as e:
    raise RuntimeError('Google Colabで実行してください。') from e

gpu_name = subprocess.check_output(
    ['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True
).strip()
gpu_mem = int(subprocess.check_output(
    ['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'], text=True
).strip())
print('GPU:', gpu_name, gpu_mem, 'MiB')
if gpu_mem < 38000:
    raise RuntimeError('本学習はA100 40GB/80GBまたはH100を使用してください。L4は固定Simulator評価向けです。')

free = shutil.disk_usage('/content').free / 1024**3
print(f'local free: {free:.1f} GiB')
if free < 45:
    raise RuntimeError('fresh runtimeで45 GiB以上の空きを確保してください。')

drive.mount('/content/drive')
try:
    token = userdata.get('HF_TOKEN')
except Exception as e:
    raise RuntimeError('Colab SecretsにHF_TOKENを登録し、Notebook accessをONにしてください。') from e
if not token:
    raise RuntimeError('HF_TOKEN is empty')
os.environ['HF_TOKEN'] = token
print('HF_TOKEN: FOUND (hidden)')

# tie-breakが再度同点だった場合だけ、結果を確認した上で以下を有効化してください。
# os.environ['SELECTED_VARIANT_OVERRIDE'] = 'V1_MULTI'
# os.environ['SELECTED_VARIANT_OVERRIDE'] = 'V2_SQRT'

print('=== PREFLIGHT: PASS ===')


In [ ]:
# 1/2 — Clone main and run the resume-safe full-training runner
import os, subprocess
from pathlib import Path

ROOT = Path('/content/parc2026')
REPO = ROOT / 'py_AI'
ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-B','main','origin/main'], check=True)
sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('repo SHA:', sha, flush=True)

runner = REPO / 'tools/colab/run_pi05_full_train_manifest.py'
if not runner.exists():
    raise RuntimeError('full-training PRをmergeしてからRun allしてください: ' + str(runner))

env = os.environ.copy()
env.update({
    'PYTHONUNBUFFERED':'1',
    'PARC_ROOT':str(ROOT),
    'PY_AI_REPO':str(REPO),
    'PARC_DRIVE_ROOT':'/content/drive/MyDrive/parc2026-cache',
})

print('=== START FULL TRAIN / RESUME ===', flush=True)
subprocess.run(['python','-u',str(runner)], cwd=str(REPO), env=env, check=True)
print('=== NOTEBOOK: PASS ===', flush=True)
